In [1]:
from ech_discovery.data.aggregates import common
from ech_discovery.notebook import *

In [2]:
def load_grace_period(summary: bool) -> pl.DataFrame:
    files = [
        file
        for role in ("tcp", "quic")
        for file in config.RAW_DATA_DIR.glob(
            f"tud/echtool/grace-period-{role}-cert-noverify/*/*"
        )
        if ("summary" in file.stem) == summary
    ]
    return (
        pl.read_ndjson(files, include_file_paths=C.filename)
        .with_columns(
            pl.col(C.filename)
            .str.split("/")
            .list.get(-2)
            .str.replace_all("_", ":")
            .str.to_datetime("%+")
            .dt.date()
            .alias(C.measurement),
            pl.col(C.ts).str.to_datetime("%+"),
        )
        .with_columns(pl.col(C.filename).str.extract("(tcp|quic)").alias(C.role))
        .filter(pl.col(C.role) == "tcp")
        .filter(pl.col(C.measurement) == pl.col(C.measurement).max())
    )


df = load_grace_period(summary=False)
df_summary = load_grace_period(summary=True)

In [3]:
# Get ECH public name
domains = pl.concat([df.select(C.domain), df_summary.select(C.domain)]).unique()
domains_and_public_name = (
    common.ech_with_dns_status(
        limit_per_location=True, regex_filter="tud/2026-07-30", limit=1
    )
    .select(C.target, C.ech_public_name)
    .join(domains.lazy(), left_on=C.target, right_on=C.domain, how="semi")
    .with_columns(
        # nordvpn returns individual response domains, fix this manually in the matching dataset
        pl.when(
            (pl.col(C.target) == "au765.proxy.nordvpn.com")
            & (pl.col(C.ech_public_name) == "bChFPfc7.example.com")
        )
        .then(pl.lit("qkNsHvVS.example.com"))
        .otherwise(C.ech_public_name)
        .alias(C.ech_public_name)
    )
    .unique()
).collect()

assert (
    domains_and_public_name.group_by(C.target)
    .agg(pl.col(C.ech_public_name).n_unique())
    .filter(pl.col(C.ech_public_name) > 1)
    .select(pl.len())
    .item()
    == 0
), "Multiple ECH public names per domain"

In [4]:
# Annotate with ECH public name
def with_public_name(frame: pl.DataFrame) -> pl.DataFrame:
    return frame.join(
        domains_and_public_name,
        left_on=C.domain,
        right_on=C.target,
        how="left",
        validate="m:1",
    )


df = with_public_name(df)
df_summary = with_public_name(df_summary)

In [5]:
failed_seeding = (
    df_summary.group_by(C.ech_public_name)
    .agg((~pl.col(C.bootstrap_ok)).all())
    .filter(pl.col(C.bootstrap_ok))
    .with_columns(
        pl.lit(Strings.not_available).alias(Strings.ech_key_grace_period),
        pl.lit(Strings.not_available).alias(Strings.ech_key_rotation_interval),
    )
)

In [6]:
# 2 source: probe and 2 source: grease
check_for_completeness = pl.len().over(C.domain, C.source) == 2
under_test = (
    df.filter(
        pl.col(C.event).str.contains("config declared no longer acc")
        | pl.col(C.event).str.contains("config under test obtained"),
    )
    .sort(C.ts)
    # Ignore the first two, those are dependent on when our measurement starts
    .filter(pl.int_range(pl.len()).over(C.domain) >= 2)
    .with_columns(
        (pl.col(C.ech_config_list).drop_nulls().n_unique())
        .over(C.domain)
        .alias(C.ech_config_list_n_unique)
    )
    .filter(check_for_completeness)
    # ignore transient failures (same configuration before and after failed conn)
    .filter(pl.col(C.ech_config_list_n_unique) == 2)
    .sort(C.ts)
).with_columns((pl.len() / 2).over(C.domain).alias(C.samples))
under_test

timestamp,domain,source,event,ech_config_list,filename,measurement,role,ech_public_name,ech_config_list-n_unique,samples
"datetime[μs, UTC]",str,str,str,str,str,date,str,str,u32,f64
2026-08-10 11:52:19.098238 UTC,"""us8360.nordvpn.com""","""grease""","""config under test obtained (sa…","""AE/+DQBLAQAgACBJ0e8CZLUa5wDMqe…","""/mnt/data01/artifacts-active-e…",2026-08-10,"""tcp""","""XCaS3Wum.example.com""",2,1.0
2026-08-10 11:53:19.116608 UTC,"""ca1624.nordvpn.com""","""grease""","""config under test obtained (sa…","""AE/+DQBLAQAgACDQXxhnRXPxR0cYr/…","""/mnt/data01/artifacts-active-e…",2026-08-10,"""tcp""","""wgV7mbzp.example.com""",2,2.0
2026-08-10 13:02:12.971599 UTC,"""zvcdn.de""","""grease""","""config under test obtained""","""AEP+DQA/pwAgACBOqMqI/73u0iiH/7…","""/mnt/data01/artifacts-active-e…",2026-08-10,"""tcp""","""arm.zvcdn.de""",2,2.0
…,…,…,…,…,…,…,…,…,…,…
2026-08-15 15:07:16.083019 UTC,"""64e.org""","""probe""","""config declared no longer acce…",null,"""/mnt/data01/artifacts-active-e…",2026-08-10,"""tcp""","""64e.org""",2,2.0
2026-08-16 02:02:11.377678 UTC,"""nghttp2.org""","""probe""","""config declared no longer acce…",null,"""/mnt/data01/artifacts-active-e…",2026-08-10,"""tcp""","""nghttp2.org""",2,2.0


In [7]:
rotated_out = (
    df.filter(pl.col(C.source) == EchtoolSource.retry)
    .sort(C.ech_config_list, C.ts)
    .group_by(C.domain, C.ech_config_list)
    .agg(pl.col(C.ts).min())
    .sort(C.ts)
    .with_columns(pl.col(C.ts).shift(-1).over(C.domain).alias(C.rotated_out))
    .drop(C.ts)
)
rotated_out

domain,ech_config_list,rotated_out
str,str,"datetime[μs, UTC]"
"""apps-403392095241187.apps.fbsb…","""AsH+DQBBBQAgACCEpikd9ey1gwO/Xp…",null
"""40sec.me""","""AMX+DQA/hAAgACDXU+RNrdoLdDK+eh…",2026-08-10 11:24:07.745982 UTC
"""1punjabisexvideo.com""","""AEX+DQBBdAAgACBWC2CHb5OuEHNJoa…",2026-08-10 12:00:07.660158 UTC
…,…,…
"""nghttp2.org""","""AD7+DQA6AQAgACAdYefN42E1IBHQtz…",null
"""megavideo.dev""","""AED+DQA8twAgACB4utE59G6JnpDMrA…",null


In [8]:
fallback_grace_period_per_public_name = (
    df.filter(
        pl.col(C.source) == EchtoolSource.probe,
        pl.col(C.event) == "accepted",
    )
    .group_by(C.domain, C.ech_public_name, C.ech_config_list)
    .agg(pl.col(C.ts).max().alias(C.last_accepted))
    .filter(pl.col(C.ech_config_list).n_unique().over(C.domain) > 1)
    .join(rotated_out, on=[C.domain, C.ech_config_list], how="inner")
    .filter(pl.col(C.last_accepted) > pl.col(C.rotated_out))
    .group_by(C.ech_public_name)
    .agg(pl.lit(Strings.latex_cmark).alias(Strings.ech_key_grace_period))
)
fallback_grace_period_per_public_name

ech_public_name,Grace Period [h]
str,str
"""cover.haka.se""","""\cmark"""
"""cloudflare-ech.com""","""\cmark"""
"""64e.org""","""\cmark"""
…,…
"""arm.zvcdn.de""","""\cmark"""
"""nghttp2.org""","""\cmark"""


In [9]:
fallback_grace_observed_but_not_inferred_per_public_name = (
    df.filter(
        (pl.col(C.source) == EchtoolSource.retry),
        pl.col(C.event).str.contains("ffers a new"),
    )
    .group_by(C.domain, C.ech_public_name)
    .agg(pl.col(C.ech_config_list).drop_nulls().n_unique())
    .filter(pl.col(C.ech_config_list) > 1)
    .group_by(C.ech_public_name)
    .agg(pl.lit(Strings.latex_cmark).alias(Strings.ech_key_rotation_interval))
)
fallback_grace_observed_but_not_inferred_per_public_name

ech_public_name,Rotation Interval [h]
str,str
"""tls-outer.browserleaks.com""","""\cmark"""
"""cover.haka.se""","""\cmark"""
"""nghttp2.org""","""\cmark"""
…,…
"""64e.org""","""\cmark"""
"""google.com""","""\cmark"""


In [10]:
grace_period = with_public_name(
    under_test.join(
        rotated_out, on=[C.domain, C.ech_config_list], how="left", validate="m:1"
    )
    # Fill when  the config was first retrieved
    .with_columns(pl.col(C.rotated_out).fill_null(pl.col(C.ts)))
    .with_columns(
        (
            (
                pl.col(C.rotated_out) - pl.col(C.rotated_out).shift(1).over(C.domain)
            ).dt.total_seconds()
            / 60
            / 60
        ).alias(C.grace_period)
    )
    .filter(pl.col(C.source) == EchtoolSource.probe)
    .group_by(C.domain)
    .agg(
        pl.col(C.grace_period).mean().round().cast(pl.Int64).alias(C.mean_grace_period),
        pl.col(C.grace_period).std().alias(C.std_grace_period),
    )
    .sort(C.mean_grace_period)
)
grace_period

domain,mean_grace_period,std_grace_period,ech_public_name
str,i64,f64,str
"""censorship.no""",0,0.001179,"""farmal.in"""
"""zoeyvid.de""",1,0.659966,"""arm.zvcdn.de"""
"""zvcdn.de""",1,0.660948,"""arm.zvcdn.de"""
…,…,…,…
"""tls.browserleaks.com""",24,0.0,"""tls-outer.browserleaks.com"""
"""nghttp2.org""",24,0.000196,"""nghttp2.org"""


In [11]:
grace_per_public_name = (
    grace_period.group_by(C.ech_public_name)
    .agg(pl.col(C.mean_grace_period).unique(), pl.col(C.std_grace_period).unique())
    .with_columns(
        pl.col(C.std_grace_period).list.max(),
        pl.when(pl.col(C.mean_grace_period).list.len() > 1)
        .then(
            pl.col(C.mean_grace_period).list.min().cast(pl.String)
            + pl.lit("-")
            + pl.col(C.mean_grace_period).list.max().cast(pl.String)
        )
        .otherwise(pl.col(C.mean_grace_period).list.max().cast(pl.String))
        .alias(C.mean_grace_period),
    )
)
grace_per_public_name

ech_public_name,mean_grace_period,std_grace_period
str,str,f64
"""farmal.in""","""0""",0.001179
"""cloudflare-ech.com""","""4""",0.000982
"""64e.org""","""1""",1.085409
…,…,…
"""nghttp2.org""","""24""",0.000196
"""tls-outer.browserleaks.com""","""24""",0.0


In [12]:
rotations = (
    df.filter(
        (
            (pl.col(C.source) == EchtoolSource.dns)
            & (pl.col(C.event) == "DNS advertises a new ECH config")
        )
        | (
            (pl.col(C.source) == EchtoolSource.retry)
            & (pl.col(C.event) == "server offers a new ECH config via RetryConfigs")
        )
    )
    .sort(C.ts)
    .with_columns(
        (
            (
                pl.col(C.ts) - pl.col(C.ts).shift().over(C.source, C.domain)
            ).dt.total_seconds()
            / 60
            / 60
        ).alias(C.rotation)
    )
    .group_by(C.domain, C.source)
    .agg(
        pl.col(C.rotation).mean().round().cast(pl.Int64()).name.suffix("-mean"),
        pl.col(C.rotation).std().name.suffix("-std"),
    )
    .pivot(C.source, index=C.domain)
    .pipe(with_public_name)
    .join(
        grace_period.drop(C.ech_public_name), on=C.domain, how="left", validate="1:1"
    )
)

assert (
    rotations[C.ech_public_name].null_count() == 0
), "domain without an ECH public name"

rotations

domain,rotation-mean_retry,rotation-mean_dns,rotation-std_retry,rotation-std_dns,ech_public_name,mean_grace_period,std_grace_period
str,i64,i64,f64,f64,str,i64,f64
"""oiranomori.jp""",1,1,0.000856,0.087421,"""cloudflare-ech.com""",4,0.000786
"""hentaihub.org""",1,1,0.000779,0.162865,"""cloudflare-ech.com""",4,0.000982
"""1win-f10.top""",1,1,0.000744,0.080394,"""cloudflare-ech.com""",4,0.000982
…,…,…,…,…,…,…,…
"""pwacart.com""",1,null,0.000143,null,"""cloudflare-ech.com""",4,0.0
"""strip2.xxx""",null,null,null,null,"""usuarioporno.com""",null,null


In [13]:
def fill_from(
    frame: pl.DataFrame, fallback: pl.DataFrame, columns: dict[str, str]
) -> pl.DataFrame:
    return (
        frame.join(fallback, on=C.ech_public_name, how="full", coalesce=True)
        .with_columns(
            pl.coalesce(pl.col(target).cast(pl.String), fallback_column).alias(target)
            for target, fallback_column in columns.items()
        )
        .drop(columns.values())
    )


rotations_by_public_name = (
    rotations.group_by(C.ech_public_name)
    .agg(
        pl.col("^rotation-mean.*$").unique().drop_nulls(),
        pl.col("^rotation-std.*$").unique().drop_nulls(),
    )
    .with_columns(
        pl.col("^rotation-std.*$").list.max(),
        pl.col("^rotation-mean.*$").list.get(0, null_on_oob=True).cast(pl.Int64),
    )
    .join(grace_per_public_name, on=C.ech_public_name, how="full", coalesce=True)
    .pipe(
        fill_from,
        fallback_grace_observed_but_not_inferred_per_public_name,
        {C.rotation_mean_retry: Strings.ech_key_rotation_interval},
    )
    .pipe(
        fill_from,
        fallback_grace_period_per_public_name,
        {C.mean_grace_period: Strings.ech_key_grace_period},
    )
    .pipe(
        fill_from,
        failed_seeding,
        {
            C.rotation_mean_retry: Strings.ech_key_rotation_interval,
            C.mean_grace_period: Strings.ech_key_grace_period,
        },
    )
)

assert (
    rotations_by_public_name[C.ech_public_name].null_count() == 0
), "null-keyed deployment"
assert rotations_by_public_name[C.ech_public_name].is_unique().all(), "duplicate deployment"

rotations_by_public_name

ech_public_name,rotation-mean_retry,rotation-mean_dns,rotation-std_retry,rotation-std_dns,mean_grace_period,std_grace_period,bootstrap_ok
str,str,i64,f64,f64,str,f64,bool
"""ow4URJQ4.example.com""","""n/a""",null,null,null,"""n/a""",null,true
"""i0qLuVW1.example.com""","""n/a""",null,null,null,"""n/a""",null,true
"""XDBIBEdW.example.com""","""n/a""",null,null,null,"""n/a""",null,true
…,…,…,…,…,…,…,…
"""yandex.video""",null,null,null,null,null,null,null
"""nghttp2.org""","""24""",24,0.000317,0.023439,"""24""",0.000196,null


In [14]:
rotations_by_public_name.write_parquet(
    config.PROCESSED_DATA_DIR
    / "echtool_config_rotation_by_public_name_graceperiod_cert_noverify.pq.zst"
)

In [15]:
rotations.write_parquet(
    config.PROCESSED_DATA_DIR / "echtool_config_rotation_graceperiod_cert_noverify.pq.zst"
)